### Given a sim, plot the number of reactions used and kinetic target scatter

In [1]:
import pandas as pd
import os, dill
import numpy as np
import cvxpy as cp
from typing import Iterable, Optional, Mapping, cast, Any
from plotly import graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
from yaml import emit

os.chdir(os.path.expanduser('~/dev/vEcoli'))

%load_ext autoreload

In [2]:
def load_sim(
        time_num: int,
        date: str,
        experiment_name: str,
        condition: str,
        experiment_type: str,
):
    # --- Load Sim ---
    time = str(time_num)
    entry = f'{experiment_name}_{time}_{date}'
    folder_path = f'out/{experiment_type}/{condition}/{entry}/'

    output = np.load(folder_path + '0_output.npy',allow_pickle='TRUE').item()
    output = output['agents']['0']
    fba = output['listeners']['fba_results']
    bulk = pd.DataFrame(output['bulk'])
    f = open(folder_path + 'agent_steps.pkl', 'rb')
    agent = dill.load(f)
    f.close()

    metabolism = agent['ecoli-metabolism-redux-classic']

    return fba, bulk, metabolism, output

In [7]:
def plot_rxn_usage(
        fba: dict,
        metabolism: object,
        output: dict,
        show_plot: bool = True,
        save_plot: bool = True,
        save_path: Optional[str] = None,
        renderer: Optional[str] = None,
):

    # --- Load Data ---
    counts_to_molar = output['listeners']['enzyme_kinetics']['counts_to_molar']

    reaction_names = metabolism.reaction_names
    sim_flux = pd.DataFrame(fba['estimated_fluxes'], columns=reaction_names).mul(counts_to_molar, axis=0)

    # --- Report Minimum Flux ---
    min_reaction_flux = sim_flux.replace(0, np.nan).min()
    min_sim_flux = min_reaction_flux.min()
    print(f'The all time non-zero minimum flux across all reactions is {min_sim_flux:0.3e} mmol/L/s')

    # --- Prepare Data for Plot ---
    # calculate the number of reactions used in the simulation
    non_zero_flux = sim_flux.gt(min_sim_flux, axis=0).sum()
    num_reactions_always_used = np.sum(non_zero_flux == time_num)
    num_reactions_never_used = np.sum(non_zero_flux == 0)
    num_reactions_occasionally_used = np.sum((non_zero_flux > 0) & (non_zero_flux < time_num))

    assert num_reactions_always_used + num_reactions_never_used + num_reactions_occasionally_used == len(reaction_names)

    # plot pie chart for distribution of reactions
    df_plot = pd.DataFrame({
        'Usage': ['Always Used', 'Never Used', 'Occasionally Used'],
        'Count': [num_reactions_always_used, num_reactions_never_used, num_reactions_occasionally_used]
    })
    fig = px.pie(df_plot, values='Count', names='Usage',
                 title=f'Distribution of Reaction Usage in Simulation - {experiment_name}',
                 color_discrete_sequence=px.colors.qualitative.Pastel,
                 labels={'Usage': 'Reaction Usage', 'Count': 'Number of Reactions'})
    fig.update_traces(textposition='inside', textinfo='percent+label')
    fig.update_layout(showlegend=False,
                      template="plotly_white",
                      paper_bgcolor='rgba(255, 0, 0, 0)',
                      plot_bgcolor='rgba(255, 0, 0, 0)',
                      title_font_size=20,
                      )

    # --- Show and/or Save Plot ---
    if show_plot:
        fig.show(renderer=renderer)

    if save_plot:

        assert save_path is not None, "Please provide a save path to save the plot."

        save_path = f'{save_path}figures/'

        if not os.path.exists(save_path):
            os.makedirs(save_path)
            print(f"Directory '{save_path}' created.")

        fig.write_image(f'{save_path}reaction_usage_pie_chart.png', width=600, height=600, scale=3)
    return

In [8]:
def plot_kinetic_scatter(
        fba: dict,
        metabolism: object,
        output: dict,
        show_plot: bool = True,
        save_plot: bool = True,
        save_path: Optional[str] = None,
        renderer: Optional[str] = None,
):
    # --- Load Data ---
    counts_to_molar = output['listeners']['enzyme_kinetics']['counts_to_molar']

    reaction_names = metabolism.reaction_names
    kinetic_reactions = metabolism.kinetic_constraint_reactions
    sim_flux = pd.DataFrame(fba['estimated_fluxes'], columns=reaction_names).mul(counts_to_molar, axis=0)
    kinetic_target = pd.DataFrame(fba["target_kinetic_fluxes"], columns=kinetic_reactions).mul(counts_to_molar, axis=0)

    # --- Process Data for Plot ---
    sim_flux_mean = sim_flux.mean(axis=0).to_frame(name="mean_flux")
    kinetic_target_mean = kinetic_target.mean(axis=0)

    sim_flux_mean["kinetic"] = [kinetic_target_mean[r] if r in kinetic_reactions else False for r in reaction_names]

    # --- Plot Scatter ---
    # plot scatter of kinetic target vs mean flux for kinetic reactions
    kinetic_reactions_flux = sim_flux_mean[sim_flux_mean['kinetic'] != False]

    fig = px.scatter(kinetic_reactions_flux, x='kinetic', y='mean_flux',
                     title='Mean Flux vs Kinetic Target for Kinetic Reactions',
                     labels={'kinetic': 'Mean Kinetic Target (mmol/L/s)', 'mean_flux': 'Mean Estimated Flux (mmol/L/s)'},
                     hover_data=['kinetic', 'mean_flux', kinetic_reactions_flux.index],
                     color_discrete_sequence=px.colors.qualitative.Pastel)
    # add y=x line
    fig.add_trace(go.Scatter(x=[0, kinetic_reactions_flux['kinetic'].max()], y=[0, kinetic_reactions_flux['kinetic'].max()],
                              mode='lines', name='y=x', line=dict(color=px.colors.qualitative.Pastel[1], dash='dash')))
    fig.update_layout(template="plotly_white",
                       showlegend=False,
                       paper_bgcolor='rgba(255, 0, 0, 0)',
                       plot_bgcolor='rgba(255, 0, 0, 0)',
                       title_font_size=20,
                       )

    # --- Show and/or Save Plot ---
    if show_plot:
        fig.show(renderer=renderer)

    if save_plot:

        assert save_path is not None, "Please provide a save path to save the plot."

        save_path = f'{folder}figures/'

        if not os.path.exists(save_path):
            os.makedirs(save_path)
            print(f"Directory '{save_path}' created.")

        fig.write_image(f'{save_path}kinetic_target_scatter.png', width=800, height=600, scale=3)

    return

In [14]:
def plot_flux_distribution(
        fba: dict,
        metabolism: object,
        output: dict,
        show_plot: bool = True,
        save_plot: bool = True,
        save_path: Optional[str] = None,
        renderer: Optional[str] = None,
):
    # --- Load Data ---
    counts_to_molar = output['listeners']['enzyme_kinetics']['counts_to_molar']

    reaction_names = metabolism.reaction_names
    sim_flux = pd.DataFrame(fba['estimated_fluxes'], columns=reaction_names).mul(counts_to_molar, axis=0)

    # --- Process Data for Plot ---
    sim_flux_mean = sim_flux.mean(axis=0).to_frame(name="mean_flux")

    # --- Plot Distribution ---
    fig = px.histogram(sim_flux_mean, x='mean_flux', nbins=50, title='Distribution of Mean Fluxes Across Reactions',
                     labels={'mean_flux': 'Mean Estimated Flux (mmol/L/s)'},
                     color_discrete_sequence=px.colors.qualitative.Pastel)
    fig.update_layout(template="plotly_white",
                      showlegend=False,
                      paper_bgcolor='rgba(255, 0, 0, 0)',
                      plot_bgcolor='rgba(255, 0, 0, 0)',
                      title_font_size=20,
                      )

    # --- Show and/or Save Plot ---
    if show_plot:
        fig.show(renderer=renderer)

    if save_plot:

        assert save_path is not None, "Please provide a save path to save the plot."

        save_path = f'{folder}figures/'

        if not os.path.exists(save_path):
            os.makedirs(save_path)
            print(f"Directory '{save_path}' created.")

        fig.write_image(f'{save_path}mean_flux_distribution.png', width=800, height=600, scale=3)

In [17]:
# import sim
time_num = 600
date = '2026-01-22'
experiment_name = 'homeostatic_only'
condition = 'basal'
experiment_type = 'objective_weight'

fba, bulk, metabolism, output = load_sim(time_num, date, experiment_name, condition, experiment_type)

In [18]:
plot_flux_distribution(fba, metabolism, output, show_plot = True, renderer = 'browser', save_plot = False)

In [12]:
folder = f'out/{experiment_type}/{condition}/{experiment_name}_{time_num}_{date}/'
plot_rxn_usage(fba, metabolism, output, show_plot = True, renderer = 'browser', save_plot = False, save_path = folder)

The all time non-zero minimum flux across all reactions is 1.382e-06 mmol/L/s


In [150]:
folder = f'out/{experiment_type}/{condition}/{experiment_name}_{time_num}_{date}/'
plot_rxn_usage(fba, metabolism, output, show_plot = True, renderer = 'browser', save_plot = True, save_path = folder)

The all time non-zero minimum flux across all reactions is 1.381e-06 mmol/L/s
Directory 'out/objective_weight/basal/homeo_kinetics_1.64E-3_600_2026-02-12/figures/' created.


In [147]:
folder = f'out/{experiment_type}/{condition}/{experiment_name}_{time_num}_{date}/'
plot_rxn_usage(fba, metabolism, output, show_plot = True, renderer = 'browser', save_plot = True, save_path = folder)

The all time non-zero minimum flux across all reactions is 1.381e-06 mmol/L/s
Directory 'out/objective_weight/basal/homeo_efficiency_2.34e-05_600_2026-01-27/figures/' created.


In [129]:
plot_rxn_usage(fba, metabolism, output, show_plot = True, renderer = 'browser', save_plot = False)

The all time non-zero minimum flux across all reactions is 1.379e-06 mmol/L/s


In [13]:
plot_kinetic_scatter(fba, metabolism, output, show_plot = True, renderer = 'browser', save_plot = False, save_path = folder)

In [158]:
1.381e-06/output['listeners']['enzyme_kinetics']['counts_to_molar'][-1]

1.0001324654128871

In [159]:
output['listeners']['enzyme_kinetics']['counts_to_molar'][-1]

1.3808170894941184e-06